# Model Training — FDA Adverse Event Signal Detector

This notebook trains and evaluates a classifier predicting whether an adverse event report is marked serious, using the cleaned, feature-engineered FDA dataset. It picks up directly from the findings in `eda.ipynb`, which found the target imbalanced (79% serious, 21% not serious), patient age weakly related to seriousness, and a meaningful, drug-dependent variation in serious-report rates across the 15 target drugs.

The baseline model is logistic regression, starting with numeric features (reaction_count, drug_count, patientonsetage_years, patientsex), with drug identity added as a richer feature set once the baseline pipeline is confirmed working. Given the class imbalance found in EDA, evaluation will use precision, recall, and a confusion matrix rather than accuracy alone.

In [2]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
conn = sqlite3.connect("../data/fda_adverse_events.db")
df = pd.read_sql_query("SELECT * FROM reports", conn)

## Handling Missing Values

reaction_count and drug_count are complete for every report, but patientonsetage_years is missing for 4,089 reports and patientsex for 428. Rather than dropping this data, values will be filled in so the full dataset can be used for training. checking patientsex's existing coding scheme first, since "Unknown" (0) may already be a valid category rather than something to invent.

In [4]:
df[["reaction_count", "drug_count", "patientonsetage_years", "patientsex"]].isnull().sum()

reaction_count              0
drug_count                  0
patientonsetage_years    4089
patientsex                428
dtype: int64

In [6]:
df["patientsex"].value_counts(dropna=False)

patientsex
2      12892
1       9377
0        476
NaN      428
Name: count, dtype: int64

In [7]:
df["patientsex"].dtype

<StringDtype(na_value=nan)>

In [8]:
df["patientonsetage_years"] = df["patientonsetage_years"].fillna(df["patientonsetage_years"].median())
df["patientsex"] = df["patientsex"].fillna("0")

In [9]:
df[["reaction_count", "drug_count", "patientonsetage_years", "patientsex"]].isnull().sum()

reaction_count           0
drug_count               0
patientonsetage_years    0
patientsex               0
dtype: int64

## Building X and y

With missing values handled, building the feature matrix (X) and target (y) for the baseline model, using the full dataset of 23,173 reports.

In [11]:
X = df[["reaction_count", "drug_count", "patientonsetage_years", "patientsex"]]
y = (df["serious"] == "1").astype(int)

## Train/Test Split

Splitting the data into a training set (used to fit the model) and a held-out test set (used only to evaluate it afterward), so performance is measured on data the model has never seen. Given the class imbalance found in EDA (79% serious, 21% not serious), the split is stratified to preserve that same ratio in both the training and test sets.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [15]:
X_train.shape


(18538, 4)

In [16]:
X_test.shape

(4635, 4)

In [18]:
y_train.value_counts(normalize=True)

serious
1    0.790646
0    0.209354
Name: proportion, dtype: float64

In [19]:
y_test.value_counts(normalize=True)

serious
1    0.790723
0    0.209277
Name: proportion, dtype: float64